In [4]:
#all imports
import pandas as pd
import numpy as np
from pathlib import Path

# Set the actual location of your CyberGuard project
PROJECT_DIR = Path(r"C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD")

DATA_DIR = PROJECT_DIR / "malicious_url" / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "malicious_url" / "models"
RESULTS_DIR = PROJECT_DIR / "malicious_url" / "results"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

phiusiil_path = DATA_DIR / "phiusiil_clean.csv"
uci_path = DATA_DIR / "uci_clean.csv"

phiusiil_df = pd.read_csv(phiusiil_path)
uci_df = pd.read_csv(uci_path)

print("PhiUSIIL shape:", phiusiil_df.shape)
print("UCI shape:", uci_df.shape)
print("PhiUSIIL columns:", phiusiil_df.columns.tolist())
print("UCI columns:", uci_df.columns.tolist())

PhiUSIIL shape: (235370, 57)
UCI shape: (11055, 33)
PhiUSIIL columns: ['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label', 'cyberguard

In [6]:
#phisUSIL features group
from sklearn.model_selection import train_test_split

phi_exclude = [
    "FILENAME",
    "URL",
    "Domain",
    "TLD",
    "Title",
    "label",
    "cyberguard_label"
]

X_phi = phiusiil_df.drop(
    columns=phi_exclude,
    errors="ignore"
)

X_phi = X_phi.select_dtypes(include=np.number)
y_phi = phiusiil_df["cyberguard_label"].astype(int)

X_phi_train, X_phi_test, y_phi_train, y_phi_test = train_test_split(
    X_phi,
    y_phi,
    test_size=0.20,
    random_state=42,
    stratify=y_phi
)

print("Training:", X_phi_train.shape)
print("Testing:", X_phi_test.shape)

Training: (188296, 50)
Testing: (47074, 50)


In [7]:
#UCI features group
uci_exclude = [
    "id",
    "Result",
    "cyberguard_label"
]

X_uci = uci_df.drop(
    columns=uci_exclude,
    errors="ignore"
)

X_uci = X_uci.select_dtypes(include=np.number)
y_uci = uci_df["cyberguard_label"].astype(int)

X_uci_train, X_uci_test, y_uci_train, y_uci_test = train_test_split(
    X_uci,
    y_uci,
    test_size=0.20,
    random_state=42,
    stratify=y_uci
)

print("Training:", X_uci_train.shape)
print("Testing:", X_uci_test.shape)

Training: (8844, 30)
Testing: (2211, 30)


In [20]:
experiments = [
    (
        "PhiUSIIL",
        X_phi_train,
        X_phi_test,
        y_phi_train,
        y_phi_test
    ),
    (
        "UCI",
        X_uci_train,
        X_uci_test,
        y_uci_train,
        y_uci_test
    )
]

print("Experiments:", len(experiments))
print("Datasets:", [exp[0] for exp in experiments])

Experiments: 2
Datasets: ['PhiUSIIL', 'UCI']


In [21]:
#training and comparing two models
def evaluate_model(model, X_train, X_test, y_train, y_test, name):
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1]
    ).ravel()

    results = {
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "f1_score": f1_score(
            y_test, predictions, zero_division=0
        ),
        "false_positive_rate": (
            fp / (fp + tn) if (fp + tn) else 0
        ),
        "false_negative_rate": (
            fn / (fn + tp) if (fn + tp) else 0
        )
    }

    print(f"\n===== {name} =====")

    print(classification_report(
        y_test,
        predictions,
        labels=[0, 1],
        target_names=["Legitimate", "Malicious"],
        zero_division=0
    ))

    print("Confusion Matrix:")
    print(confusion_matrix(
        y_test, predictions, labels=[0, 1]
    ))

    return results, model

In [18]:
print("Evaluation function cell executed successfully.")
print(evaluate_model)

Evaluation function cell executed successfully.
<function evaluate_model at 0x0000018F6AB65A80>


In [22]:
print("Experiments:", len(experiments))
print("Datasets:", [exp[0] for exp in experiments])

Experiments: 2
Datasets: ['PhiUSIIL', 'UCI']


In [23]:
results = []
trained_models = {}

for dataset_name, X_train, X_test, y_train, y_test in experiments:

    print(f"\nStarting dataset: {dataset_name}", flush=True)

    models = {
        "RandomForest": RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),
        "HistGradientBoosting": HistGradientBoostingClassifier(
            max_iter=150,
            random_state=42
        )
    }

    for model_name, model in models.items():

        print(f"Training {model_name}...", flush=True)

        result, trained_model = evaluate_model(
            model,
            X_train,
            X_test,
            y_train,
            y_test,
            f"{dataset_name}_{model_name}"
        )

        results.append({
            "dataset": dataset_name,
            **result
        })

        trained_models[f"{dataset_name}_{model_name}"] = trained_model

        print(f"Finished {model_name}", flush=True)

results_df = pd.DataFrame(results)

print("\nMODEL COMPARISON")
display(results_df.sort_values("f1_score", ascending=False))


Starting dataset: PhiUSIIL
Training RandomForest...

===== PhiUSIIL_RandomForest =====
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     26970
   Malicious       1.00      1.00      1.00     20104

    accuracy                           1.00     47074
   macro avg       1.00      1.00      1.00     47074
weighted avg       1.00      1.00      1.00     47074

Confusion Matrix:
[[26970     0]
 [    0 20104]]
Finished RandomForest
Training HistGradientBoosting...

===== PhiUSIIL_HistGradientBoosting =====
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     26970
   Malicious       1.00      1.00      1.00     20104

    accuracy                           1.00     47074
   macro avg       1.00      1.00      1.00     47074
weighted avg       1.00      1.00      1.00     47074

Confusion Matrix:
[[26970     0]
 [    0 20104]]
Finished HistGradientBoosting

Starting dataset: UCI
Training 

,dataset,model,accuracy,precision,recall,f1_score,false_positive_rate,false_negative_rate
0,PhiUSIIL,PhiUSIIL_RandomForest,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000
1,PhiUSIIL,PhiUSIIL_HistGradientBoosting,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000
2,UCI,UCI_RandomForest,0.976029,0.978328,0.967347,0.972807,0.017059,0.032653
3,UCI,UCI_HistGradientBoosting,0.971506,0.976116,0.959184,0.967576,0.018684,0.040816
